In [2]:
from openaq import OpenAQ

client = OpenAQ(api_key="YOUR_OPENAQ_API_KEY")#Your own API key from OpenAQ website

datetime_from = "2024-01-01T00:00:00Z"
datetime_to   = "2025-11-30T23:59:59Z"
sensor_id = 6516  # e.g. 6516

all_measurements = []
page = 1
limit = 1000

while True:
    measurements = client.measurements.list(
        sensors_id=sensor_id,
        datetime_from=datetime_from,
        datetime_to=datetime_to,
        limit=limit,
        page=page,
    )
    
    results = measurements.results  # list[Measurement]
    if not results:
        break   # stop when a page is empty

    all_measurements.extend(results)
    page += 1

client.close()

print("Total measurements collected:", len(all_measurements))


Total measurements collected: 13618


In [3]:
import json
import pandas as pd

records = []
for m in all_measurements:
    rec = {
        "value": m.value,
        "parameter": m.parameter.name,          # "no2"
        "units": m.parameter.units,             # "µg/m³"
        "datetime_from_utc": m.period.datetime_from.utc,
        "datetime_to_utc": m.period.datetime_to.utc,
        "location_id": 3063,
        "location_name": "Essen-Ost Steeler Straße",
        "sensor_id": sensor_id,
    }
    records.append(rec)

print("Rows:", len(records))
print(records[0])  # quick check


Rows: 13618
{'value': 5.816, 'parameter': 'no2', 'units': 'µg/m³', 'datetime_from_utc': '2024-01-01T00:00:00Z', 'datetime_to_utc': '2024-01-01T01:00:00Z', 'location_id': 3063, 'location_name': 'Essen-Ost Steeler Straße', 'sensor_id': 6516}


In [4]:
# save as JSON corpus
with open("no2_essen_2025-08_11_raw.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

# load to DataFrame and save CSV
df = pd.DataFrame(records)
df.to_csv("no2_essen_2025-08_11_raw.csv", index=False)

df.head()


,value,parameter,units,datetime_from_utc,datetime_to_utc,location_id,location_name,sensor_id
0,5.816,no2,µg/m³,2024-01-01T00:00:00Z,2024-01-01T01:00:00Z,3063,Essen-Ost Steeler Straße,6516
1,3.816,no2,µg/m³,2024-01-01T01:00:00Z,2024-01-01T02:00:00Z,3063,Essen-Ost Steeler Straße,6516
2,2.329,no2,µg/m³,2024-01-01T02:00:00Z,2024-01-01T03:00:00Z,3063,Essen-Ost Steeler Straße,6516
3,1.464,no2,µg/m³,2024-01-01T03:00:00Z,2024-01-01T04:00:00Z,3063,Essen-Ost Steeler Straße,6516
4,2.096,no2,µg/m³,2024-01-01T04:00:00Z,2024-01-01T05:00:00Z,3063,Essen-Ost Steeler Straße,6516
